# Classificação de diabetes com XGBoost e Azure Machine Learning (dataset NIDDK)

<mark>Este notebook é didático e não deve ser utilizado para realizar diagnósticos</mark>

Este notebook reproduz o experimento em `02-xgboost-niddk-diabetes.ipynb`, mas registra rastreabilidade, artefatos e versões no Azure Machine Learning. Ele é independente do `03`: precisa somente do Parquet produzido por `01-prepare-niddk-diabetes.ipynb`.

O treinamento continua local: este notebook usa os dados preparados por `01-prepare-niddk-diabetes.ipynb` para treinar e avaliar um classificador binário **XGBoost**. O Azure Machine Learning recebe os registros da execução, o modelo e os data assets versionados dos conjuntos de treino e teste.

## Preparar o Azure Machine Learning local

Execute em um terminal separado, partindo da raiz do repositório, para instalar as dependências e autenticar sua conta Azure:

```powershell
cd ml
uv sync --locked
az login
```

Se tiver mais de uma assinatura, selecione a que contém o workspace:

```powershell
az account set --subscription "<sua-subscription-id>"
```

Crie sua configuração local a partir do exemplo. O arquivo `.env` é ignorado pelo Git e não deve conter senhas, chaves ou tokens:

```powershell
Copy-Item .env.example .env
```

Preencha `AZURE_SUBSCRIPTION_ID`, `AZURE_RESOURCE_GROUP` e `AZURE_ML_WORKSPACE` com o workspace que você criou.

O notebook usa `DefaultAzureCredential`, que reutiliza a autenticação de `az login`. Ele não cria recursos Azure automaticamente. Antes de registrar dados ou modelos, a célula de conexão exibe a assinatura, o resource group e o workspace selecionados.

Os dados de treino e teste são publicados como versões dos data assets `niddk-diabetes-train` e `niddk-diabetes-test`. O notebook `06` poderá usar as versões mais recentes desses assets para gerar o relatório Responsible AI.

## Importações e caminhos do projeto

### Bibliotecas

Usamos `mlflow.sklearn.autolog()` com o URI de tracking do Azure Machine Learning para acompanhar o treinamento local do pipeline. O MLflow registra automaticamente parâmetros do estimador, métricas de treino, dataset de treino, modelo, assinatura e exemplo de entrada. As métricas de teste e os artefatos específicos do projeto continuam explícitos.

In [ ]:
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory
import hashlib
import os
import platform

from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from mlflow import MlflowClient
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")


def find_project_root() -> Path:
    """Retorna a raiz do repositório quando o notebook é executado neste projeto."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "ml" / "pyproject.toml").is_file() and (
            candidate / "README.md"
        ).is_file():
            return candidate
    raise FileNotFoundError("Abra este notebook dentro do repositório do projeto.")


def required_environment_variable(name: str) -> str:
    """Lê uma configuração obrigatória do workspace Azure ML."""
    value = os.getenv(name)
    if not value:
        raise RuntimeError(
            f"A variável {name} não está definida. Copie ml/.env.example para ml/.env, "
            "preencha as configurações do seu workspace e execute esta célula novamente."
        )
    return value

### Variáveis

In [ ]:
PROJECT_ROOT = find_project_root()

DATA_PATH = PROJECT_ROOT / "data" / "niddk-diabetes" / "diabetes.parquet"
TARGET_COLUMN = "diabetes"
IDENTIFIER_COLUMN = "id_paciente"

RANDOM_STATE = 42
TEST_SIZE = 0.20
EXPERIMENT_NAME = "niddk-diabetes-xgboost"
REGISTERED_MODEL_NAME = "niddk-diabetes-xgboost"
TRAIN_DATA_ASSET_NAME = "niddk-diabetes-train"
TEST_DATA_ASSET_NAME = "niddk-diabetes-test"
NOTEBOOK_NAME = "04-azureml-niddk-diabetes.ipynb"

MODEL_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "n_estimators": 200,
    "max_depth": 3,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

## Conectar ao Azure Machine Learning e definir o experimento

Um **experimento** reúne execuções comparáveis. Cada **run** representa uma execução local de treinamento e avaliação, registrada no workspace Azure Machine Learning. O **Model Registry** mantém versões do modelo registrado.

In [ ]:
load_dotenv(PROJECT_ROOT / "ml" / ".env")

subscription_id = required_environment_variable("AZURE_SUBSCRIPTION_ID")
resource_group = required_environment_variable("AZURE_RESOURCE_GROUP")
workspace_name = required_environment_variable("AZURE_ML_WORKSPACE")

credential = DefaultAzureCredential()
ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name,
)
workspace = ml_client.workspaces.get(name=workspace_name)

mlflow.set_tracking_uri(workspace.mlflow_tracking_uri)
mlflow.set_registry_uri(workspace.mlflow_tracking_uri)
experiment = mlflow.set_experiment(EXPERIMENT_NAME)

print("Azure Machine Learning selecionado:")
print(f"- Assinatura: {subscription_id}")
print(f"- Resource group: {resource_group}")
print(f"- Workspace: {workspace_name}")
print(f"- MLflow: {mlflow.get_tracking_uri()}")
print(f"- Experimento: {EXPERIMENT_NAME} (ID {experiment.experiment_id})")

## Carregar o dataset preparado

O notebook de preparação traduz as colunas de origem para português e as salva em Parquet. Antes de prosseguir, verificamos se as colunas de alvo e identificador esperadas estão presentes.

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset preparado não encontrado em {DATA_PATH}. "
        "Execute primeiro 01-prepare-niddk-diabetes.ipynb."
    )

data = pd.read_parquet(DATA_PATH)
required_columns = {TARGET_COLUMN, IDENTIFIER_COLUMN}
missing_columns = required_columns.difference(data.columns)
if missing_columns:
    raise ValueError(f"O dataset não possui as colunas obrigatórias: {sorted(missing_columns)}")

print(f"Dimensões do dataset: {data.shape}")

In [ ]:
print(f"Distribuição das classes:\n{data[TARGET_COLUMN].value_counts(normalize=True).sort_index()}")

In [ ]:
display(data.head())

## Separar os conjuntos em variáveis de entrada `X` e alvo `y`

O identificador do paciente não é uma característica clínica e, portanto, não participa do treinamento. Ele é preservado em `data` e nos data assets para permitir rastreabilidade dos registros.

In [ ]:
feature_columns = [
    column
    for column in data.columns
    if column not in {TARGET_COLUMN, IDENTIFIER_COLUMN}
]
if not feature_columns:
    raise ValueError("Não há colunas disponíveis para treinamento após remover o alvo e identificador.")

X = data[feature_columns]
y = data[TARGET_COLUMN]

print(f"Características ({len(feature_columns)}): {feature_columns}")

## Criar os conjuntos de treino e teste

A divisão é estratificada para preservar a proporção de classes. Os índices retornados permitem publicar os registros completos — incluindo identificador e alvo — como data assets versionados no Azure Machine Learning.

In [ ]:
train_index, test_index = train_test_split(
    data.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

train_data = data.loc[train_index].reset_index(drop=True)
test_data = data.loc[test_index].reset_index(drop=True)
X_train = train_data[feature_columns]
y_train = train_data[TARGET_COLUMN]
X_test = test_data[feature_columns]
y_test = test_data[TARGET_COLUMN]

print(f"Treino: {X_train.shape}; teste: {X_test.shape}")

## Treinar, avaliar e registrar uma execução

Antes do treinamento, publicamos os conjuntos efetivamente usados como novas versões dos data assets. O treinamento permanece local, enquanto o MLflow registra a rodada, seus parâmetros, métricas, gráficos, relatórios e modelo no Azure Machine Learning.

O nome e a versão dos data assets usados são registrados como tags da run. O notebook `06` poderá selecionar a última versão de cada asset para a análise Responsible AI.

A versão do modelo é criada automaticamente pelo `autolog()`. Além dos artefatos do modelo, salvamos a matriz de confusão, a importância das variáveis, relatórios, metadados do split, versões de pacotes, o lockfile e uma cópia deste notebook.

As métricas de teste são: `test_accuracy`, `test_precision`, `test_recall`, `test_f1` e `test_roc_auc`. Elas avaliam a previsão da classe positiva (`diabetes=1`) no limiar padrão do modelo.

A importância das características é específica deste modelo e não estabelece causalidade clínica.

Abaixo, cada data asset é criado a partir de um arquivo Parquet temporário. O Azure ML copia o arquivo para o datastore do workspace antes que o diretório temporário seja removido.

In [ ]:
dataset_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()

with TemporaryDirectory() as temporary_directory:
    temporary_directory_path = Path(temporary_directory)
    train_path = temporary_directory_path / "niddk-diabetes-train.parquet"
    test_path = temporary_directory_path / "niddk-diabetes-test.parquet"
    train_data.to_parquet(train_path, index=False)
    test_data.to_parquet(test_path, index=False)

    train_asset = ml_client.data.create_or_update(
        Data(
            name=TRAIN_DATA_ASSET_NAME,
            path=str(train_path),
            type=AssetTypes.URI_FILE,
            description="Conjunto de treino do experimento XGBoost NIDDK para fins didáticos.",
        )
    )
    test_asset = ml_client.data.create_or_update(
        Data(
            name=TEST_DATA_ASSET_NAME,
            path=str(test_path),
            type=AssetTypes.URI_FILE,
            description="Conjunto de teste do experimento XGBoost NIDDK para fins didáticos.",
        )
    )

print(f"Data asset de treino: {train_asset.name}:{train_asset.version}")
print(f"Data asset de teste: {test_asset.name}:{test_asset.version}")

mlflow.sklearn.autolog(
    registered_model_name=REGISTERED_MODEL_NAME,
    log_input_examples=True,
    log_model_signatures=True,
    log_models=True,
    log_datasets=True,
    log_post_training_metrics=False,
    silent=False,
)

pipeline = Pipeline([( "classifier", XGBClassifier(**MODEL_PARAMS))])

with mlflow.start_run(run_name="xgboost-baseline") as run:
    run_id = run.info.run_id
    mlflow.set_tags(
        {
            "dataset_name": DATA_PATH.name,
            "dataset_sha256": dataset_sha256,
            "notebook_name": NOTEBOOK_NAME,
            "python_version": platform.python_version(),
            "train_data_asset_name": train_asset.name,
            "train_data_asset_version": str(train_asset.version),
            "train_data_asset_id": train_asset.id,
            "test_data_asset_name": test_asset.name,
            "test_data_asset_version": str(test_asset.version),
            "test_data_asset_id": test_asset.id,
        }
    )
    mlflow.log_params(
        {
            "test_size": TEST_SIZE,
            "random_state": RANDOM_STATE,
            "feature_count": len(feature_columns),
        }
    )

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_probability = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred, zero_division=0),
        "test_recall": recall_score(y_test, y_pred, zero_division=0),
        "test_f1": f1_score(y_test, y_pred, zero_division=0),
        "test_roc_auc": roc_auc_score(y_test, y_probability),
    }
    mlflow.log_metrics(metrics)
    mlflow.log_dict(
        classification_report(y_test, y_pred, output_dict=True, zero_division=0),
        "reports/classification_report.json",
    )
    mlflow.log_dict(
        {
            "random_state": RANDOM_STATE,
            "test_size": TEST_SIZE,
            "feature_columns": feature_columns,
            "train_rows": len(train_data),
            "test_rows": len(test_data),
        },
        "metadata/data_split.json",
    )
    mlflow.log_dict(
        {
            "parquet_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
            "parquet_sha256": dataset_sha256,
            "train_data_asset": {"name": train_asset.name, "version": train_asset.version, "id": train_asset.id},
            "test_data_asset": {"name": test_asset.name, "version": test_asset.version, "id": test_asset.id},
        },
        "metadata/data_assets.json",
    )
    mlflow.log_dict(
        {
            package: version(package)
            for package in ("azure-ai-ml", "azure-identity", "azureml-mlflow", "mlflow", "pandas", "scikit-learn", "xgboost")
        },
        "metadata/package_versions.json",
    )

    lockfile = PROJECT_ROOT / "ml" / "uv.lock"
    if lockfile.is_file():
        mlflow.log_artifact(str(lockfile), artifact_path="metadata")

    figure, axis = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axis, colorbar=False)
    axis.set_title("Matriz de confusão - conjunto de teste")
    figure.tight_layout()
    mlflow.log_figure(figure, "plots/confusion_matrix.png")
    plt.close(figure)

    feature_importance = pd.Series(
        pipeline.named_steps["classifier"].feature_importances_, index=feature_columns
    ).sort_values()
    figure, axis = plt.subplots(figsize=(8, 5))
    feature_importance.plot.barh(ax=axis)
    axis.set_title("Importância das características")
    axis.set_xlabel("Importância")
    figure.tight_layout()
    mlflow.log_figure(figure, "plots/feature_importance.png")
    plt.close(figure)

    notebook_path = PROJECT_ROOT / "ml" / "notebooks" / NOTEBOOK_NAME
    if notebook_path.is_file():
        mlflow.log_artifact(str(notebook_path), artifact_path="metadata")

print(f"Run registrada: {run_id}")
print(pd.Series(metrics).round(4))
print(classification_report(y_test, y_pred, digits=3, zero_division=0))

## Consultar a versão registrada

Como `autolog()` não retorna um `ModelInfo`, buscamos a versão do modelo cujo `run_id` corresponde à execução atual. Isso evita escolher simplesmente a versão mais recente registrada por outra rodada.

In [ ]:
client = MlflowClient()
model_versions = client.search_model_versions(f"run_id = '{run_id}'")
registered_versions = [
    model_version
    for model_version in model_versions
    if model_version.name == REGISTERED_MODEL_NAME
]
if not registered_versions:
    raise RuntimeError(
        "Era esperada uma versão criada pelo autolog nesta run. "
        "Verifique o acesso ao Azure Machine Learning e execute a célula de treinamento novamente."
    )

registered_version = max(registered_versions, key=lambda item: int(item.version))
model_version = registered_version.version
MODEL_URI = f"models:/{REGISTERED_MODEL_NAME}/{model_version}"

print(f"Nome: {REGISTERED_MODEL_NAME}")
print(f"Versão: {model_version}; status: {registered_version.status}")
print(f"URI para carregar: {MODEL_URI}")

## Recarregar pelo Azure Machine Learning e verificar a inferência

O artefato é baixado para `outputs/models/`, criada automaticamente. Conferimos classes e probabilidades para todo o teste, verificando que o modelo recarregado reproduz o modelo local. Usamos o flavor scikit-learn para preservar `predict_proba`.

In [ ]:
DOWNLOAD_DIR = PROJECT_ROOT / "outputs" / "models"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

loaded_pipeline = mlflow.sklearn.load_model(MODEL_URI, dst_path=str(DOWNLOAD_DIR))
np.testing.assert_array_equal(loaded_pipeline.predict(X_test), y_pred)
np.testing.assert_allclose(
    loaded_pipeline.predict_proba(X_test)[:, 1],
    y_probability,
    rtol=1e-12,
    atol=1e-12,
)

example = X_test.head(3)
display(
    example.assign(
        classe_prevista=loaded_pipeline.predict(example),
        probabilidade_classe_1=loaded_pipeline.predict_proba(example)[:, 1],
    )
)
print("Modelo registrado e recarregado: previsões e probabilidades conferidas.")